# QubitBoost SDK Integration

qb-compiler works standalone as an open-source quantum compiler, but it also integrates with the
[QubitBoost SDK](https://qubitboost.io) for access to seven specialised quantum flight control gates.

This notebook covers:
- Automatic circuit type detection (`detect_circuit_type`)
- Gate recommendation engine (`recommend_gates`)
- The `GATE_REGISTRY`: metadata for all 7 gates
- `GateRecommendation` fields and how to interpret them
- `QubitBoostExecutor` for running circuits through gates
- Building a pre-flight pipeline that chains qb-compiler viability checks with QubitBoost gates

In [1]:
from qb_compiler.integrations.qubitboost import (
    GATE_REGISTRY,
    Confidence,
    GateRecommendation,
    detect_circuit_type,
    is_sdk_available,
    recommend_gates,
)
from qb_compiler import QBCircuit, check_viability, BACKEND_CONFIGS

## 1. Check SDK availability

The QubitBoost SDK is an optional dependency. `is_sdk_available()` returns `True` if the
`qubitboost-sdk` package is installed. All detection and recommendation features work without
the SDK: you only need it for actual gate execution via `QubitBoostExecutor`.

In [2]:
sdk_installed = is_sdk_available()
print(f"QubitBoost SDK installed: {sdk_installed}")

if not sdk_installed:
    print("\nGate recommendations and circuit detection work without the SDK.")
    print("Install for execution: pip install qubitboost-sdk")

QubitBoost SDK installed: True


## 2. Circuit type detection

`detect_circuit_type()` examines a circuit's name and gate structure to classify it as
one of four types: `qaoa`, `vqe`, `qec`, or `general`.

It returns a `(circuit_type, confidence)` tuple where confidence is `HIGH`, `MEDIUM`, or `LOW`.

### Confidence levels

| Level | How detected | Effect on recommendations |
|-------|-------------|---------------------------|
| **HIGH** | Circuit name contains `qaoa`, `vqe`, `qec`, etc. | Full validated claims shown |
| **MEDIUM** | Gate structure matches known patterns | Gates shown as "May be eligible" |
| **LOW** | No clear signal | Only universal gates shown |

In [3]:
from qiskit import QuantumCircuit

# QAOA-like circuit: name gives HIGH confidence
qaoa_circ = QuantumCircuit(4, name="qaoa_maxcut_p3")
for i in range(4):
    qaoa_circ.h(i)
for i in range(3):
    qaoa_circ.rzz(0.5, i, i + 1)
for i in range(4):
    qaoa_circ.rx(0.7, i)

circuit_type, confidence = detect_circuit_type(qaoa_circ)
print(f"Circuit: {qaoa_circ.name}")
print(f"Detected type: {circuit_type}")
print(f"Confidence: {confidence.value}")

Circuit: qaoa_maxcut_p3
Detected type: qaoa
Confidence: high


In [4]:
# VQE-like circuit: structural detection gives MEDIUM confidence
vqe_circ = QuantumCircuit(4, name="my_experiment")
for i in range(4):
    vqe_circ.ry(0.3, i)
for i in range(3):
    vqe_circ.cx(i, i + 1)
for i in range(4):
    vqe_circ.rz(0.5, i)
    vqe_circ.ry(0.2, i)

circuit_type, confidence = detect_circuit_type(vqe_circ)
print(f"Circuit: {vqe_circ.name}")
print(f"Detected type: {circuit_type}")
print(f"Confidence: {confidence.value}")

Circuit: my_experiment
Detected type: vqe
Confidence: medium


In [5]:
# QEC circuit: name match gives HIGH confidence
qec_circ = QuantumCircuit(9, name="surface_code_d3")
for i in range(0, 8, 2):
    qec_circ.cx(i, i + 1)
qec_circ.measure_all()

circuit_type, confidence = detect_circuit_type(qec_circ)
print(f"Circuit: {qec_circ.name}")
print(f"Detected type: {circuit_type}")
print(f"Confidence: {confidence.value}")

Circuit: surface_code_d3
Detected type: qec
Confidence: high


In [6]:
# General circuit: no clear pattern, LOW confidence
general_circ = QuantumCircuit(3, name="test")
general_circ.h(0)
general_circ.cx(0, 1)
general_circ.cx(1, 2)
general_circ.measure_all()

circuit_type, confidence = detect_circuit_type(general_circ)
print(f"Circuit: {general_circ.name}")
print(f"Detected type: {circuit_type}")
print(f"Confidence: {confidence.value}")

Circuit: test
Detected type: general
Confidence: low


## 3. The GATE_REGISTRY

All seven QubitBoost gates are described in `GATE_REGISTRY`, a dict mapping gate name to
`GateInfo` metadata. Each entry contains the gate's headline, hardware-validated claim,
execution phase, and which circuit types it applies to.

In [7]:
print(f"Total gates in registry: {len(GATE_REGISTRY)}\n")

for name, info in GATE_REGISTRY.items():
    print(f"{name:14s}  phase={info.phase:6s}  types={info.circuit_types}")
    print(f"{'':14s}  {info.headline}")
    print(f"{'':14s}  Validated: {info.validated_claim} {info.qualifier}")
    print(f"{'':14s}  Requires high confidence: {info.requires_high_confidence}")
    print()

Total gates in registry: 7

OptGate         phase=during  types=('qaoa',)
                Adaptive QAOA shot reduction
                Validated: adaptive shot allocation on validated QAOA workloads
                Requires high confidence: True

ChemGate        phase=during  types=('vqe',)
                VQE operator preselection
                Validated: 32-42% fewer evaluations on validated VQE workflows
                Requires high confidence: True

TomoGate        phase=pre     types=('qaoa', 'vqe', 'qec', 'general')
                Pre-flight state fidelity certification
                Validated: Pre-flight certification for supported circuit types
                Requires high confidence: False

LiveGate        phase=during  types=('qaoa', 'vqe', 'general')
                Real-time doom detection
                Validated: Abort failing runs early on supported backends
                Requires high confidence: False

SafetyGate      phase=pre     types=('qec',)
            

### Gate phases

Gates operate at different points in the quantum execution lifecycle:

| Phase | Gates | Purpose |
|-------|-------|----------|
| **pre** | TomoGate, SafetyGate | Pre-flight checks before QPU submission |
| **during** | OptGate, ChemGate, GuardGate, LiveGate | Active during circuit execution |
| **post** | ShotValidator | Post-execution result verification |

In [8]:
# Group gates by phase
from collections import defaultdict

by_phase = defaultdict(list)
for name, info in GATE_REGISTRY.items():
    by_phase[info.phase].append(name)

for phase in ["pre", "during", "post"]:
    gates = by_phase.get(phase, [])
    print(f"{phase:6s}: {', '.join(gates)}")

pre   : TomoGate, SafetyGate
during: OptGate, ChemGate, LiveGate, GuardGate
post  : ShotValidator


## 4. Gate recommendations

`recommend_gates()` takes a circuit type and confidence level, and returns a list of
`GateRecommendation` objects sorted by execution phase.

### GateRecommendation fields

| Field | Description |
|-------|-------------|
| `gate` | Gate name (e.g. `"OptGate"`) |
| `status` | `"Eligible"`, `"May be eligible"`, or `"Available"` |
| `headline` | Short description of what the gate does |
| `validated_claim` | Hardware-validated performance number (or `None`) |
| `qualifier` | Scope of the validated claim (or `None`) |
| `phase` | Execution phase: `"pre"`, `"during"`, or `"post"` |

In [9]:
# Recommendations for a QAOA circuit at HIGH confidence
recs = recommend_gates("qaoa", Confidence.HIGH)

print("QAOA circuit (HIGH confidence):")
print(f"  {len(recs)} gates recommended\n")

for r in recs:
    print(f"  [{r.phase:6s}] {r.gate:14s}: {r.status}")
    print(f"           {r.headline}")
    if r.validated_claim:
        print(f"           Validated: {r.validated_claim} {r.qualifier}")
    print()

QAOA circuit (HIGH confidence):
  5 gates recommended

  [pre   ] TomoGate      : Available
           Pre-flight state fidelity certification

  [during] OptGate       : Eligible
           Adaptive QAOA shot reduction
           Validated: adaptive shot allocation on validated QAOA workloads

  [during] GuardGate     : Eligible
           QAOA quality assurance
           Validated: QAOA quality assurance on validated QAOA workloads

  [during] LiveGate      : Available
           Real-time doom detection

  [post  ] ShotValidator : Available
           Result integrity verification



In [10]:
# Compare: same circuit type at LOW confidence: specialised gates are filtered out
recs_low = recommend_gates("qaoa", Confidence.LOW)

print("QAOA circuit (LOW confidence):")
print(f"  {len(recs_low)} gates recommended (specialised gates hidden)\n")

for r in recs_low:
    print(f"  [{r.phase:6s}] {r.gate:14s}: {r.status}: {r.headline}")

QAOA circuit (LOW confidence):
  3 gates recommended (specialised gates hidden)

  [pre   ] TomoGate      : Available: Pre-flight state fidelity certification
  [during] LiveGate      : Available: Real-time doom detection
  [post  ] ShotValidator : Available: Result integrity verification


In [11]:
# Full pipeline: detect type, then get recommendations
for circ in [qaoa_circ, vqe_circ, qec_circ, general_circ]:
    ctype, conf = detect_circuit_type(circ)
    recs = recommend_gates(ctype, conf)
    gate_names = [r.gate for r in recs]
    print(f"{circ.name:25s}  type={ctype:7s}  conf={conf.value:6s}  gates={gate_names}")

qaoa_maxcut_p3             type=qaoa     conf=high    gates=['TomoGate', 'OptGate', 'GuardGate', 'LiveGate', 'ShotValidator']
my_experiment              type=vqe      conf=medium  gates=['TomoGate', 'ChemGate', 'LiveGate', 'ShotValidator']
surface_code_d3            type=qec      conf=high    gates=['SafetyGate', 'TomoGate', 'ShotValidator']
test                       type=general  conf=low     gates=['TomoGate', 'LiveGate', 'ShotValidator']


## 5. QubitBoostExecutor

`QubitBoostExecutor` bridges qb-compiler with the QubitBoost SDK for actual gate execution.
It requires `pip install qubitboost-sdk`.

```python
from qb_compiler.integrations.qubitboost import QubitBoostExecutor

executor = QubitBoostExecutor(backend="ibm_fez")

# OptGate: adaptive QAOA shot reduction
result = executor.execute_optgate(circuit, problem_graph=G, shots=4096)

# ChemGate: VQE operator preselection
result = executor.execute_chemgate(circuit, hamiltonian=H)

# TomoGate: pre-flight fidelity certification
result = executor.execute_tomogate(circuit)

# SafetyGate: QEC trust scoring
result = executor.execute_safetygate(syndrome_history, distance=7)
```

In [12]:
# QubitBoostExecutor will raise ImportError if SDK is not installed
if is_sdk_available():
    from qb_compiler.integrations.qubitboost import QubitBoostExecutor
    print("QubitBoostExecutor available")
else:
    print("QubitBoostExecutor requires qubitboost-sdk")
    print("Install with: pip install qubitboost-sdk")
    print("\nAll other features in this notebook work without it.")

QubitBoostExecutor available


## 6. Pre-flight pipeline: qb-compiler viability + QubitBoost gates

The recommended workflow combines qb-compiler's viability check with QubitBoost gate
recommendations into a single pre-flight pipeline. This tells you:
1. Whether the circuit will produce meaningful results on the target backend
2. Which QubitBoost gates can improve your results

In [13]:
def preflight_pipeline(circuit, backend="ibm_fez"):
    """Run a complete pre-flight check with gate recommendations."""
    # Step 1: Detect circuit type
    ctype, conf = detect_circuit_type(circuit)
    print(f"Circuit: {circuit.name}")
    print(f"Type: {ctype} (confidence: {conf.value})")
    print()

    # Step 2: Check viability on target backend
    result = check_viability(circuit, backend=backend)
    print(f"Backend: {backend}")
    print(f"Status: {result.status}")
    print(f"Estimated fidelity: {result.estimated_fidelity:.4f}")
    print(f"Signal/noise: {result.signal_to_noise:.1f}x")
    if result.cost_estimate_usd is not None:
        print(f"Cost (4096 shots): ${result.cost_estimate_usd:.4f}")
    print()

    # Step 3: Gate recommendations
    recs = recommend_gates(ctype, conf)
    if recs:
        print("Recommended QubitBoost gates:")
        for r in recs:
            claim = f": {r.validated_claim}" if r.validated_claim else ""
            print(f"  {r.gate:14s} [{r.phase}] {r.status}{claim}")
    print()

    # Step 4: Actionable suggestions
    if result.suggestions:
        print("Suggestions:")
        for s in result.suggestions:
            print(f"  - {s}")

    return result, recs


# Run the pipeline on a QAOA circuit
viability, gates = preflight_pipeline(qaoa_circ, backend="ibm_fez")

Circuit: qaoa_maxcut_p3
Type: qaoa (confidence: high)



/home/dministrator/miniconda3/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.1) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Backend: ibm_fez
Status: VIABLE
Estimated fidelity: 0.9900
Signal/noise: 15.8x
Cost (4096 shots): $0.6554

Recommended QubitBoost gates:
  TomoGate       [pre] Available
  OptGate        [during] Eligible: adaptive shot allocation
  GuardGate      [during] Eligible: QAOA quality assurance
  LiveGate       [during] Available
  ShotValidator  [post] Available

Suggestions:
  - Circuit looks good: proceed with execution.
  - Calibration snapshot is 90 days old; the estimate reflects that date. Pass fresh backend_props or set QBC_CALIBRATION_DIR for current numbers.
